In [2]:
import pandas as pd
import re
import spacy
import joblib

# Load cleaned dataset
df = pd.read_csv("clean_jobs.csv")

# Same 100-job evaluation sample used in Day 15
test_df = df.sample(
    100,
    random_state=42
).copy()

test_df = test_df[
    ["job_title", "clean_description"]
].reset_index(drop=True)

print("Evaluation dataset loaded")
print("Number of jobs:", len(test_df))

Evaluation dataset loaded
Number of jobs: 100


In [3]:
skill_list = [
    "python",
    "java",
    "c++",
    "javascript",
    "typescript",
    "sql",
    "mysql",
    "postgresql",
    "mongodb",
    "power bi",
    "tableau",
    "excel",
    "pandas",
    "numpy",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "machine learning",
    "deep learning",
    "natural language processing",
    "aws",
    "azure",
    "gcp",
    "spark",
    "hadoop",
    "docker",
    "kubernetes",
    "git"
]


def get_actual_skills(text):

    text = text.lower()
    found = []

    for skill in skill_list:
        if skill in text:
            found.append(skill)

    return sorted(set(found))


test_df["actual_skills"] = test_df[
    "clean_description"
].apply(get_actual_skills)

print("Reference skills recreated")

Reference skills recreated


In [5]:
def dictionary_extract(text):
    text = text.lower()
    found = []

    for skill in skill_list:
        if skill in text:
            found.append(skill)

    return sorted(set(found))


test_df["dictionary_skills"] = test_df[
    "clean_description"
].apply(dictionary_extract)

print("Dictionary predictions created")

Dictionary predictions created


In [6]:
patterns = {
    "python": r"\bpython(?:\s*3)?(?:\s+programming)?\b",
    "java": r"\bjava\b",
    "c++": r"\bc\+\+\b",
    "javascript": r"\bjavascript\b",
    "typescript": r"\btypescript\b",
    "sql": r"\bsql\b",
    "mysql": r"\bmysql\b",
    "postgresql": r"\b(?:postgres|postgresql)\b",
    "mongodb": r"\b(?:mongodb|mongo)\b",
    "power bi": r"\bpower\s*bi\b",
    "tableau": r"\btableau\b",
    "excel": r"\b(?:excel|ms\s+excel)\b",
    "pandas": r"\bpandas\b",
    "numpy": r"\bnumpy\b",
    "scikit-learn": r"\b(?:scikit[- ]learn|sklearn)\b",
    "tensorflow": r"\btensorflow\b",
    "pytorch": r"\bpytorch\b",
    "machine learning": r"\bmachine\s+learning\b",
    "deep learning": r"\bdeep\s+learning\b",
    "natural language processing": r"\bnatural\s+language\s+processing\b",
    "aws": r"\baws\b",
    "azure": r"\bazure\b",
    "gcp": r"\b(?:gcp|google\s+cloud)\b",
    "spark": r"\b(?:spark|apache\s+spark)\b",
    "hadoop": r"\b(?:hadoop|apache\s+hadoop)\b",
    "docker": r"\bdocker\b",
    "kubernetes": r"\b(?:kubernetes|k8s)\b",
    "git": r"\bgit\b"
}


def regex_extract(text):
    found = []

    for skill, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            found.append(skill)

    return sorted(set(found))


test_df["regex_skills"] = test_df[
    "clean_description"
].apply(regex_extract)

print("Regex predictions created")

Regex predictions created


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english",
    ngram_range=(1, 3)
)

tfidf_matrix = tfidf.fit_transform(
    test_df["clean_description"]
)

terms = tfidf.get_feature_names_out()


def tfidf_extract(text, top_n=20):

    vector = tfidf.transform([text])
    scores = vector.toarray()[0]

    ranked_indices = scores.argsort()[::-1]

    top_terms = []

    for i in ranked_indices[:top_n]:
        if scores[i] > 0:
            top_terms.append(terms[i].lower())

    found = []

    for skill in skill_list:
        if skill in top_terms:
            found.append(skill)

    return sorted(set(found))


test_df["tfidf_skills"] = test_df[
    "clean_description"
].apply(tfidf_extract)

print("TF-IDF predictions created")

TF-IDF predictions created


In [8]:
nlp = spacy.load("en_core_web_sm")


def ner_extract(text):

    doc = nlp(text)
    found = []

    for ent in doc.ents:

        entity = ent.text.lower().strip()

        if entity in skill_list:
            found.append(entity)

    return sorted(set(found))


test_df["ner_skills"] = test_df[
    "clean_description"
].apply(ner_extract)

print("NER predictions created")

NER predictions created


In [9]:
ml_model = joblib.load("skill_classifier.pkl")


def ml_extract(text):

    found = []
    text_lower = text.lower()

    for skill in skill_list:

        if skill in text_lower:

            prediction = ml_model.predict([skill])[0]

            if prediction in [
                "SKILL",
                "CLOUD",
                "DATABASE",
                "BI_TOOL",
                "TECHNOLOGY"
            ]:
                found.append(skill)

    return sorted(set(found))


test_df["ml_skills"] = test_df[
    "clean_description"
].apply(ml_extract)

print("ML predictions created")

ML predictions created


In [10]:
from transformers import pipeline

transformer_ner = pipeline(
    "ner",
    model="./skill_bert_model_final",
    tokenizer="./skill_bert_model_final",
    aggregation_strategy="simple"
)


def transformer_extract(text):

    results = transformer_ner(text)

    found = []

    for entity in results:

        word = entity["word"].lower().strip()
        word = word.replace("##", "")

        if not any(c.isalnum() for c in word):
            continue

        for skill in skill_list:

            skill_clean = skill.replace(" ", "")

            if word == skill_clean:
                found.append(skill)

    return sorted(set(found))


test_df["transformer_skills"] = test_df[
    "clean_description"
].apply(transformer_extract)

print("Transformer predictions created")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Transformer predictions created


In [11]:
test_df[
    [
        "job_title",
        "actual_skills",
        "dictionary_skills",
        "regex_skills",
        "tfidf_skills",
        "ner_skills",
        "ml_skills",
        "transformer_skills"
    ]
].head(10)

,job_title,actual_skills,dictionary_skills,regex_skills,tfidf_skills,ner_skills,ml_skills,transformer_skills
0,Backend Developer,"[docker, postgresql, sql]","[docker, postgresql, sql]","[docker, postgresql]",[postgresql],[],"[docker, postgresql, sql]","[docker, postgresql]"
1,Data Scientist,"[aws, machine learning, numpy, pandas, sql]","[aws, machine learning, numpy, pandas, sql]","[aws, machine learning, numpy, pandas, scikit-...",[numpy],[],"[aws, machine learning, numpy, pandas, sql]","[aws, numpy, pandas, sql]"
2,Data Analyst,"[excel, pandas, power bi, python, sql]","[excel, pandas, power bi, python, sql]","[excel, pandas, power bi, python, sql]",[],[],"[excel, pandas, power bi, python, sql]","[excel, pandas, python, sql]"
3,QA Engineer,"[git, sql]","[git, sql]","[git, sql]",[],[],"[git, sql]","[git, sql]"
4,Business Analyst,"[excel, power bi, sql, tableau]","[excel, power bi, sql, tableau]","[excel, power bi, sql, tableau]",[],[],"[excel, power bi, sql, tableau]","[excel, sql, tableau]"
5,Data Analyst,"[excel, pandas, python, sql, tableau]","[excel, pandas, python, sql, tableau]","[excel, pandas, python, sql, tableau]","[excel, pandas, tableau]",[],"[excel, pandas, python, sql, tableau]","[excel, pandas, python, sql, tableau]"
6,Data Scientist,"[aws, machine learning, python, sql]","[aws, machine learning, python, sql]","[aws, machine learning, python, scikit-learn, ...",[],[],"[aws, machine learning, python, sql]","[aws, python, sql]"
7,Machine Learning Engineer,"[docker, machine learning, python, pytorch, sq...","[docker, machine learning, python, pytorch, sq...","[docker, machine learning, python, pytorch, sc...","[pytorch, tensorflow]",[],"[docker, machine learning, python, pytorch, sq...","[docker, python, pytorch, sql, tensorflow]"
8,Business Analyst,"[excel, power bi, sql, tableau]","[excel, power bi, sql, tableau]","[excel, power bi, sql, tableau]",[],[],"[excel, power bi, sql, tableau]","[excel, sql, tableau]"
9,Frontend Developer,"[git, java, javascript, typescript]","[git, java, javascript, typescript]","[git, javascript, typescript]","[javascript, typescript]",[],"[git, java, javascript, typescript]","[git, javascript]"


In [12]:
error_rows = []

methods = [
    "dictionary_skills",
    "regex_skills",
    "tfidf_skills",
    "ner_skills",
    "ml_skills",
    "transformer_skills"
]

method_names = {
    "dictionary_skills": "Dictionary",
    "regex_skills": "Regex",
    "tfidf_skills": "TF-IDF",
    "ner_skills": "NER",
    "ml_skills": "ML Model",
    "transformer_skills": "Transformer"
}

for i, row in test_df.iterrows():

    actual = set(row["actual_skills"])

    for method in methods:

        predicted = set(row[method])

        # False Negatives
        for skill in actual - predicted:

            error_rows.append({
                "Job_ID": i,
                "Job_Title": row["job_title"],
                "Method": method_names[method],
                "Error_Type": "False Negative",
                "Expected_Skill": skill,
                "Predicted_Skill": "",
                "Description": f"{skill} was present but was not extracted."
            })

        # False Positives
        for skill in predicted - actual:

            error_rows.append({
                "Job_ID": i,
                "Job_Title": row["job_title"],
                "Method": method_names[method],
                "Error_Type": "False Positive",
                "Expected_Skill": "",
                "Predicted_Skill": skill,
                "Description": f"{skill} was extracted but was not present."
            })


error_df = pd.DataFrame(error_rows)

print("Total error cases:", len(error_df))

Total error cases: 759


In [13]:
error_df.head(30)

,Job_ID,Job_Title,Method,Error_Type,Expected_Skill,Predicted_Skill,Description
0,0,Backend Developer,Regex,False Negative,sql,,sql was present but was not extracted.
1,0,Backend Developer,TF-IDF,False Negative,docker,,docker was present but was not extracted.
2,0,Backend Developer,TF-IDF,False Negative,sql,,sql was present but was not extracted.
3,0,Backend Developer,NER,False Negative,postgresql,,postgresql was present but was not extracted.
4,0,Backend Developer,NER,False Negative,docker,,docker was present but was not extracted.
5,0,Backend Developer,NER,False Negative,sql,,sql was present but was not extracted.
6,0,Backend Developer,Transformer,False Negative,sql,,sql was present but was not extracted.
7,1,Data Scientist,Regex,False Positive,,scikit-learn,scikit-learn was extracted but was not present.
8,1,Data Scientist,TF-IDF,False Negative,machine learning,,machine learning was present but was not extra...
9,1,Data Scientist,TF-IDF,False Negative,aws,,aws was present but was not extracted.


In [14]:
error_summary = (
    error_df
    .groupby(["Method", "Error_Type"])
    .size()
    .reset_index(name="Count")
)

error_summary

,Method,Error_Type,Count
0,NER,False Negative,380
1,Regex,False Negative,9
2,Regex,False Positive,15
3,TF-IDF,False Negative,293
4,Transformer,False Negative,62


In [15]:
method_error_summary = (
    error_df
    .groupby("Method")
    .size()
    .reset_index(name="Total_Errors")
    .sort_values("Total_Errors")
)

method_error_summary

,Method,Total_Errors
1,Regex,24
3,Transformer,62
2,TF-IDF,293
0,NER,380


In [16]:
pd.set_option("display.max_colwidth", 200)

error_details = []

for i, row in test_df.iterrows():

    actual = set(row["actual_skills"])

    for method in methods:

        predicted = set(row[method])

        # False Negative
        for skill in actual - predicted:

            error_details.append({
                "Job_ID": i,
                "Job_Title": row["job_title"],
                "Method": method_names[method],
                "Error_Type": "False Negative",
                "Expected_Skill": skill,
                "Predicted_Skill": "",
                "Job_Description": row["clean_description"]
            })

        # False Positive
        for skill in predicted - actual:

            error_details.append({
                "Job_ID": i,
                "Job_Title": row["job_title"],
                "Method": method_names[method],
                "Error_Type": "False Positive",
                "Expected_Skill": "",
                "Predicted_Skill": skill,
                "Job_Description": row["clean_description"]
            })


error_details_df = pd.DataFrame(error_details)

print("Detailed error records:", len(error_details_df))

Detailed error records: 759


In [17]:
error_details_df[
    [
        "Job_Title",
        "Method",
        "Error_Type",
        "Expected_Skill",
        "Predicted_Skill"
    ]
].head(30)

,Job_Title,Method,Error_Type,Expected_Skill,Predicted_Skill
0,Backend Developer,Regex,False Negative,sql,
1,Backend Developer,TF-IDF,False Negative,docker,
2,Backend Developer,TF-IDF,False Negative,sql,
3,Backend Developer,NER,False Negative,postgresql,
4,Backend Developer,NER,False Negative,docker,
5,Backend Developer,NER,False Negative,sql,
6,Backend Developer,Transformer,False Negative,sql,
7,Data Scientist,Regex,False Positive,,scikit-learn
8,Data Scientist,TF-IDF,False Negative,machine learning,
9,Data Scientist,TF-IDF,False Negative,aws,


In [18]:
def classify_error(row):
    
    error_type = row["Error_Type"]
    expected = str(row["Expected_Skill"]).lower()
    predicted = str(row["Predicted_Skill"]).lower()

    # Already known from evaluation
    if error_type == "False Positive":
        return "False Positive"

    if error_type == "False Negative":
        return "False Negative"

    return "Unknown"


error_details_df["Error_Category"] = error_details_df.apply(
    classify_error,
    axis=1
)

print("Error categories assigned")

Error categories assigned


In [19]:
error_details_df[
    [
        "Job_Title",
        "Method",
        "Error_Type",
        "Expected_Skill",
        "Predicted_Skill",
        "Error_Category"
    ]
].head(30)

,Job_Title,Method,Error_Type,Expected_Skill,Predicted_Skill,Error_Category
0,Backend Developer,Regex,False Negative,sql,,False Negative
1,Backend Developer,TF-IDF,False Negative,docker,,False Negative
2,Backend Developer,TF-IDF,False Negative,sql,,False Negative
3,Backend Developer,NER,False Negative,postgresql,,False Negative
4,Backend Developer,NER,False Negative,docker,,False Negative
5,Backend Developer,NER,False Negative,sql,,False Negative
6,Backend Developer,Transformer,False Negative,sql,,False Negative
7,Data Scientist,Regex,False Positive,,scikit-learn,False Positive
8,Data Scientist,TF-IDF,False Negative,machine learning,,False Negative
9,Data Scientist,TF-IDF,False Negative,aws,,False Negative


In [20]:
error_category_summary = (
    error_details_df
    .groupby(["Method", "Error_Category"])
    .size()
    .reset_index(name="Count")
)

error_category_summary

,Method,Error_Category,Count
0,NER,False Negative,380
1,Regex,False Negative,9
2,Regex,False Positive,15
3,TF-IDF,False Negative,293
4,Transformer,False Negative,62


In [21]:
error_report = error_details_df[
    [
        "Job_ID",
        "Job_Title",
        "Method",
        "Error_Category",
        "Expected_Skill",
        "Predicted_Skill",
        "Job_Description"
    ]
].copy()

error_report["Explanation"] = error_report[
    "Error_Category"
].map({
    "False Negative":
        "The expected skill was present in the reference labels but was not extracted by the method.",

    "False Positive":
        "The method extracted a skill that was not present in the reference labels."
})

error_report.head(20)

,Job_ID,Job_Title,Method,Error_Category,Expected_Skill,Predicted_Skill,Job_Description,Explanation
0,0,Backend Developer,Regex,False Negative,sql,,we are looking for a backend developer to join our team the candidate should have experience with docker postgresql node js and rest api the role involves developing solutions analyzing requiremen...,The expected skill was present in the reference labels but was not extracted by the method.
1,0,Backend Developer,TF-IDF,False Negative,docker,,we are looking for a backend developer to join our team the candidate should have experience with docker postgresql node js and rest api the role involves developing solutions analyzing requiremen...,The expected skill was present in the reference labels but was not extracted by the method.
2,0,Backend Developer,TF-IDF,False Negative,sql,,we are looking for a backend developer to join our team the candidate should have experience with docker postgresql node js and rest api the role involves developing solutions analyzing requiremen...,The expected skill was present in the reference labels but was not extracted by the method.
3,0,Backend Developer,NER,False Negative,postgresql,,we are looking for a backend developer to join our team the candidate should have experience with docker postgresql node js and rest api the role involves developing solutions analyzing requiremen...,The expected skill was present in the reference labels but was not extracted by the method.
4,0,Backend Developer,NER,False Negative,docker,,we are looking for a backend developer to join our team the candidate should have experience with docker postgresql node js and rest api the role involves developing solutions analyzing requiremen...,The expected skill was present in the reference labels but was not extracted by the method.
5,0,Backend Developer,NER,False Negative,sql,,we are looking for a backend developer to join our team the candidate should have experience with docker postgresql node js and rest api the role involves developing solutions analyzing requiremen...,The expected skill was present in the reference labels but was not extracted by the method.
6,0,Backend Developer,Transformer,False Negative,sql,,we are looking for a backend developer to join our team the candidate should have experience with docker postgresql node js and rest api the role involves developing solutions analyzing requiremen...,The expected skill was present in the reference labels but was not extracted by the method.
7,1,Data Scientist,Regex,False Positive,,scikit-learn,we are looking for a data scientist to join our team the candidate should have experience with sql aws numpy machine learning scikit learn and pandas the role involves developing solutions analyzi...,The method extracted a skill that was not present in the reference labels.
8,1,Data Scientist,TF-IDF,False Negative,machine learning,,we are looking for a data scientist to join our team the candidate should have experience with sql aws numpy machine learning scikit learn and pandas the role involves developing solutions analyzi...,The expected skill was present in the reference labels but was not extracted by the method.
9,1,Data Scientist,TF-IDF,False Negative,aws,,we are looking for a data scientist to join our team the candidate should have experience with sql aws numpy machine learning scikit learn and pandas the role involves developing solutions analyzi...,The expected skill was present in the reference labels but was not extracted by the method.


In [22]:
with pd.ExcelWriter(
    "Error_Analysis_Report.xlsx",
    engine="openpyxl"
) as writer:

    error_report.to_excel(
        writer,
        sheet_name="Error Details",
        index=False
    )

    error_category_summary.to_excel(
        writer,
        sheet_name="Error Summary",
        index=False
    )

    method_error_summary.to_excel(
        writer,
        sheet_name="Day 16 Summary",
        index=False
    )

print("Error_Analysis_Report.xlsx created successfully")

Error_Analysis_Report.xlsx created successfully


In [23]:
report = pd.ExcelFile(
    "Error_Analysis_Report.xlsx"
)

print(report.sheet_names)

['Error Details', 'Error Summary', 'Day 16 Summary']
